# Scenario Notebook: S2: HVI-CIDNet

Pipeline Fase 1–6 untuk skenario **S2_HVI_CIDNet**.

| Fase | Deskripsi | Auto-skip? |
|------|-----------|------------|
| 1    | Data Preparation (parse, convert, build YOLO) | jika output sudah ada |
| 2    | LLIE Enhancement (menggunakan HVI_CIDNet) | jika enhanced dir sudah ada |
| 3    | YOLOv11n Training | jika best.pt sudah ada |
| 4    | Detection Evaluation (mAP, P, R) | jika metrics.json sudah ada |
| 5    | NR-IQA (NIQE, BRISQUE, LOE) | jika summary.json sudah ada |
| 6    | Latency & FLOPs | jika cache sudah ada |

Setelah **semua 4 skenario** selesai buka **comparison.ipynb**.

## 0. Setup

In [ ]:
#@title 0.1 · Mount Drive & Clone Repo
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

# ── Config ──────────────────────────────────────────────────────
QUICK_TEST  = True  # @param {type:"boolean"}
REPO_URL    = "https://github.com/Otachiking/Object-Detection-ExDARK-with-LLIE.git"
DRIVE_ROOT  = "/content/drive/MyDrive/KULIAH-S1INFORMATIKA/TA-IQBAL"

SCENARIO_KEY  = "s2_hvi_cidnet"   # DO NOT CHANGE
SCENARIO_NAME = "S2_HVI_CIDNet"  # DO NOT CHANGE

# ── Clone / pull ─────────────────────────────────────────────────
REPO_DIR = "/content/TA-IQBAL-ObjectDetectionExDARKwithLLIE"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Resetting repo to latest origin/main ...")
    subprocess.run(["git","-C",REPO_DIR,"fetch","origin"], check=True)
    subprocess.run(["git","-C",REPO_DIR,"reset","--hard","origin/main"], check=True)
else:
    import shutil
    if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
    print("Cloning repo ...")
    subprocess.run(["git","clone",REPO_URL,REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"\nScenario : {SCENARIO_NAME}")
print(f"Quick test: {QUICK_TEST}")
print(f"Drive root: {DRIVE_ROOT}")


In [ ]:
#@title 0.2 · Install Dependencies
!pip install -q ultralytics pyiqa thop fvcore scipy pandas pyyaml seaborn tqdm gdown huggingface_hub

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram  = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM     : {vram/1e9:.1f} GB")
else:
    print("⚠ No GPU — Runtime > Change runtime type > T4 GPU")


In [ ]:
#@title 0.3 · Load Configuration
from src.config import load_config, save_environment_info
from src.seed import set_global_seed

cfg = load_config(SCENARIO_KEY, quick_test=QUICK_TEST)
set_global_seed(cfg["seed"])

paths      = cfg.get("paths", {})
data_paths = paths.get("data", {})

OUTPUT_ROOT = paths.get("output_root") or paths.get("drive_root") or paths.get("project_root")
EXDARK_ROOT = paths.get("exdark_root") or data_paths.get("exdark_original")

if OUTPUT_ROOT is None: raise KeyError("Cannot resolve OUTPUT_ROOT from config")
if EXDARK_ROOT is None: raise KeyError("Cannot resolve EXDARK_ROOT from config")

cfg["paths"]["output_root"]  = OUTPUT_ROOT
cfg["paths"]["exdark_root"]  = EXDARK_ROOT
if "exdark_structure" not in cfg["paths"]:
    m = cfg.get("paths_meta", {}).get("exdark", {})
    cfg["paths"]["exdark_structure"] = {
        "images":      m.get("images_dir",      "Dataset"),
        "groundtruth": m.get("groundtruth_dir",  "Groundtruth"),
        "classlist":   m.get("classlist_file",   "Groundtruth/imageclasslist.txt"),
    }

print(f"Output root : {OUTPUT_ROOT}")
print(f"ExDark root : {EXDARK_ROOT}")
assert os.path.exists(EXDARK_ROOT), f"ExDark not found: {EXDARK_ROOT}"
print("\n✓ ExDark dataset found")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
save_environment_info(OUTPUT_ROOT)


---
## Fase 1: Data Preparation

In [ ]:
#@title Fase 1 · Data Preparation  (auto-skip if already done)
from src.data.split_dataset     import parse_split_file
from src.data.convert_exdark    import convert_exdark_to_yolo
from src.data.build_yolo_dataset import build_yolo_dataset
from src.data.validate_dataset  import validate_yolo_dataset

classlist_path = os.path.join(EXDARK_ROOT,
    cfg["paths"]["exdark_structure"]["groundtruth"], "imageclasslist.txt")
img_dir        = os.path.join(EXDARK_ROOT, cfg["paths"]["exdark_structure"]["images"])
gt_dir         = os.path.join(EXDARK_ROOT, cfg["paths"]["exdark_structure"]["groundtruth"])
split_output   = os.path.join(OUTPUT_ROOT, "splits")
labels_dir     = os.path.join(OUTPUT_ROOT, "ExDark_yolo_labels")
yolo_dir       = os.path.join(OUTPUT_ROOT, "ExDark_yolo")

# 1.1 Splits
splits = parse_split_file(classlist_path, split_output)
print(f"Splits  → Train:{splits['train']} Val:{splits['val']} Test:{splits['test']}")

# 1.2 Convert annotations
stats = convert_exdark_to_yolo(img_dir, gt_dir, labels_dir)
print(f"Labels  → {stats['total_labels']} files, {stats['total_objects']} objects")

# 1.3 Build YOLO dir + dataset.yaml
build_stats = build_yolo_dataset(img_dir, labels_dir, split_output, yolo_dir,
                                  target_size=cfg["yolo"]["imgsz"])
total = sum(s["processed"] for s in build_stats["splits"].values())
print(f"YOLO dir→ {total} images built")

# 1.4 Validate
vr = validate_yolo_dataset(yolo_dir)
print(f"Validate→ {'PASSED ✓' if vr['valid'] else 'FAILED ✗'}")
if not vr["valid"]:
    for sp, sd in vr.get("splits",{}).items():
        if sd.get("error"): print(f"  {sp}: {sd['error']}")


---
## Fase 2: Image Enhancement

In [ ]:
#@title Fase 2 · Enhancement  (auto-skip if already done)
import torch
from src.enhancement.run_enhancement import enhance_dataset, get_enhancer

enhancer_name = cfg.get("enhancer", {}).get("name", None)
assert enhancer_name and enhancer_name.lower() != "none", \
    "No enhancer configured for this scenario!"

enhanced_dir = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}")
cache_dir    = os.path.join(OUTPUT_ROOT, "model_cache")

print(f"Enhancer  : {enhancer_name}")
print(f"Output    : {enhanced_dir}")

enhancer = get_enhancer(enhancer_name, cache_dir)
enhancer.load_model()

stats = enhance_dataset(
    enhancer=enhancer,
    source_dataset_dir=yolo_dir,
    output_dir=enhanced_dir,
    yolo_labels_dir=yolo_dir,
)

print(f"\nDone → {stats['total_processed']} processed, "
      f"{stats['total_skipped']} skipped, {stats['total_failed']} failed")

del enhancer
if torch.cuda.is_available(): torch.cuda.empty_cache()


---
## Fase 3: Training

In [ ]:
#@title Fase 3 · Training  (auto-skip if best.pt exists)
from src.training.train_yolo import train_yolo, get_best_weights

data_yaml = (
    os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "dataset.yaml")
    if enhancer_name and enhancer_name.lower() != "none"
    else os.path.join(OUTPUT_ROOT, "ExDark_yolo", "dataset.yaml")
)
assert os.path.exists(data_yaml), f"dataset.yaml not found: {data_yaml}"

runs_dir = os.path.join(OUTPUT_ROOT, "runs")
result   = train_yolo(dataset_yaml=data_yaml, scenario_name=SCENARIO_NAME,
                      output_dir=runs_dir, config=cfg)

best = get_best_weights(os.path.join(runs_dir, SCENARIO_NAME))
print(f"\nbest.pt  : {best}")


---
## Fase 4: Detection Evaluation

In [ ]:
#@title Fase 4 · Detection Evaluation  (auto-skip if metrics.json exists)
from src.evaluation.eval_yolo import evaluate_yolo
import pandas as pd

run_dir      = os.path.join(OUTPUT_ROOT, "runs", SCENARIO_NAME)
weights_path = get_best_weights(run_dir)
eval_dir     = os.path.join(OUTPUT_ROOT, "evaluation", SCENARIO_NAME)

results  = evaluate_yolo(weights_path=weights_path, dataset_yaml=data_yaml,
                         output_dir=eval_dir, scenario_name=SCENARIO_NAME)
overall  = results.get("overall", {})

print(f"\n{'='*50}")
print(f"Detection Results — {SCENARIO_NAME}")
print(f"{'='*50}")
print(f"  mAP@0.5      : {overall.get('mAP_50',0):.4f}")
print(f"  mAP@0.5:0.95 : {overall.get('mAP_50_95',0):.4f}")
print(f"  Precision    : {overall.get('precision',0):.4f}")
print(f"  Recall       : {overall.get('recall',0):.4f}")

per_cls = results.get("per_class", {})
if per_cls:
    cls_rows = [{"Class": k, "AP@0.5": f"{v:.4f}"} for k, v in per_cls.items()]
    import pandas as pd
    display(pd.DataFrame(cls_rows).set_index("Class").T)


---
## Fase 5: Image Quality Metrics

In [ ]:
#@title Fase 5 · Image Quality Metrics  (auto-skip if summary.json exists)
from src.evaluation.nr_metrics import compute_nr_metrics

raw_test_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "images", "test")
test_dir = (
    os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "images", "test")
    if enhancer_name and enhancer_name.lower() != "none"
    else raw_test_dir
)
nr_dir = os.path.join(OUTPUT_ROOT, "evaluation", SCENARIO_NAME, "nr_metrics")
raw_dir_for_loe = raw_test_dir if (enhancer_name and enhancer_name.lower() != "none") else None

nr = compute_nr_metrics(images_dir=test_dir, output_dir=nr_dir,
                         scenario_name=SCENARIO_NAME, raw_images_dir=raw_dir_for_loe)
print(f"\nNR-IQA — {SCENARIO_NAME}")
print(f"  NIQE (↓)    : {nr.get('niqe_mean','N/A')}")
print(f"  BRISQUE (↓) : {nr.get('brisque_mean','N/A')}")
print(f"  LOE (↓)     : {nr.get('loe_mean','N/A')}")


---
## Fase 6: Latency & FLOPs

In [ ]:
#@title Fase 6 · Latency & FLOPs  (auto-skip if cached)
import torch
from src.evaluation.latency import measure_latency
from src.evaluation.flops   import compute_all_flops

raw_test_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "images", "test")
run_dir      = os.path.join(OUTPUT_ROOT, "runs", SCENARIO_NAME)
weights_path = get_best_weights(run_dir)
lat_dir      = os.path.join(OUTPUT_ROOT, "evaluation", SCENARIO_NAME, "latency")
flops_dir    = os.path.join(OUTPUT_ROOT, "evaluation", SCENARIO_NAME, "flops")

enhancer_obj = None
if enhancer_name and enhancer_name.lower() != "none":
    from src.enhancement.run_enhancement import get_enhancer
    cache_dir    = os.path.join(OUTPUT_ROOT, "model_cache")
    enhancer_obj = get_enhancer(enhancer_name, cache_dir)
    enhancer_obj.load_model()

lat  = measure_latency(yolo_weights=weights_path, output_dir=lat_dir,
                        scenario_name=SCENARIO_NAME, test_images_dir=raw_test_dir,
                        enhancer=enhancer_obj,
                        num_images=cfg.get("latency",{}).get("iterations",200),
                        warmup=cfg.get("latency",{}).get("warmup",50))

flops = compute_all_flops(yolo_weights=weights_path, output_dir=flops_dir,
                           scenario_name=SCENARIO_NAME,
                           enhancer_model=enhancer_obj.model if enhancer_obj else None,
                           enhancer_name=enhancer_name if enhancer_obj else None)

print(f"\nLatency — {SCENARIO_NAME}")
print(f"  T_enhance : {lat.get('T_enhance_ms_mean',0):.2f} ms")
print(f"  T_detect  : {lat.get('T_detect_ms_mean',0):.2f} ms")
print(f"  T_total   : {lat.get('T_total_ms_mean',0):.2f} ms")
print(f"\nFLOPs — {SCENARIO_NAME}")
print(f"  Enhancer  : {flops.get('enhancer',{}).get('gflops',0) or 0:.2f} GFLOPs")
print(f"  YOLO      : {flops.get('yolo',{}).get('gflops',0) or 0:.2f} GFLOPs")
print(f"  Total     : {flops.get('total_gflops',0) or 0:.2f} GFLOPs")

if enhancer_obj: del enhancer_obj
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
#@title Done — S2_HVI_CIDNet complete
print("=" * 60)
print(f"  {SCENARIO_NAME} — all Fase 1-6 complete")
print(f"  Results saved under:")
print(f"    {OUTPUT_ROOT}/evaluation/{SCENARIO_NAME}/")
print(f"    {OUTPUT_ROOT}/runs/{SCENARIO_NAME}/")
print()
print("  Next steps:")
print("    -> Run the other scenario notebooks (S2, S3, S4)")
print("    -> After all 4 scenarios done, open comparison.ipynb")
print("=" * 60)
